## Prerequisites

Run these once in a fresh Colab runtime before address parsing. Coordinate-only runs do not need the llama/model cells, but they do need the Variables and Shared Helper Functions cells below.


In [ ]:
# Llama install/check for Colab GPU address parsing.
!nvcc --version
!pip uninstall -y llama-cpp-python
!pip install -q llama-cpp-python huggingface_hub --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu125

from llama_cpp import llama_cpp
print('GPU offload support:', llama_cpp.llama_supports_gpu_offload())


In [ ]:
# Download the GGUF model used by GGUF_MODEL_PATH in the Variables cell.
!hf download hf://unsloth/gemma-4-26B-A4B-it-qat-GGUF/gemma-4-26B-A4B-it-qat-UD-Q4_K_XL.gguf


### Optional: Google Drive

Mount Drive only when you want to copy inputs, outputs, or model files between Colab and Google Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Optional model backup after running the model download cell:
# !mkdir -p /content/drive/MyDrive/LLM
# !cp /path/to/model.gguf /content/drive/MyDrive/LLM/


## 0. Variables

Edit this one cell for the input path, API keys, model path, and llama settings. Output filenames are generated automatically from `INPUT_PATH`.


In [ ]:
# Edit environment variables or this cell first, then run the helper/init/action cells below.
import os

# File path. Output files are generated automatically beside INPUT_PATH.
INPUT_PATH = os.getenv('SPREADSHEET_LLM_INPUT_PATH', '/content/all_branch_data.csv')   # Example: '/content/branches.xlsx' or '/content/branches.csv'
PSGC_PATH = os.getenv('SPREADSHEET_LLM_PSGC_PATH', '')  # Required for address parsing. Example: '/content/philippines_psgc_barangays.csv'
SHEET_NAME = 'branch'  # Used only for .xlsx files.
PROCESS_ROW_LIMIT = None  # Example: 10 for testing, None for all rows.

# Fixed required column names. These are case-sensitive.
LONGITUDE_COLUMN = 'longitude'
LATITUDE_COLUMN = 'latitude'
FULL_ADDRESS_COLUMN = 'full_address'

# Google Geocoding settings. Required only for coordinate filling or run-both.
GOOGLE_GEOCODING_API_KEY = os.getenv('GOOGLE_GEOCODING_API_KEY', '')

# Local GGUF parsing settings. Required only for address parsing or run-both.
GGUF_MODEL_PATH = os.getenv('GGUF_MODEL_PATH', '')
N_CTX = 2048
N_GPU_LAYERS = -1  # -1 means offload all possible layers to GPU.
FLASH_ATTN = False
MAX_TOKENS = 180
VERBOSE_LLAMA = False

print('Variables loaded. Next run Shared Helper Functions. If parsing addresses, run the Prerequisites cells first.')


## 0.1 Shared Helper Functions

Run once after the variables cell. This loads file, geocoding, and address parsing helpers.


In [ ]:
import csv, json, math, os, re, subprocess, sys, time
from pathlib import Path


def pip_install(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

try:
    import requests
    from openpyxl import Workbook, load_workbook
    from tqdm.auto import tqdm
except ImportError:
    pip_install(['requests', 'openpyxl', 'tqdm'])
    import requests
    from openpyxl import Workbook, load_workbook
    from tqdm.auto import tqdm

NULL_LIKE_VALUES = {'', 'none', 'null', 'nan', 'n/a', 'na', '-', '--'}
PARSED_ADDRESS_COLUMNS = ['barangay2', 'city_municipality2', 'province2', 'region2', 'zip_code2', 'country2', 'island_group2']
ADDRESS_FIELDS = ['barangay', 'city_municipality', 'province', 'zip_code', 'country', 'island_group']
OUTPUT_FIELD_COLUMNS = {'barangay': 'barangay2', 'city_municipality': 'city_municipality2', 'province': 'province2', 'region': 'region2', 'zip_code': 'zip_code2', 'country': 'country2', 'island_group': 'island_group2'}
PROVINCE_ALIASES = {'Agusan Del Sur': 'Agusan del Sur', 'Davao De Oro': 'Davao de Oro', 'Davao Del Norte': 'Davao del Norte', 'Davao Del Sur': 'Davao del Sur', 'North Cotabato': 'Cotabato', 'Surigao Del Sur': 'Surigao del Sur', 'Western Samar': 'Samar', 'Zamboanga Del Norte': 'Zamboanga del Norte'}
SPECIAL_PROVINCE_REGIONS = {'Metro Manila': 'National Capital Region (NCR)'}
SPECIAL_REGION_CODES = {'National Capital Region (NCR)': 'NCR', 'Cordillera Administrative Region (CAR)': 'CAR', 'Bangsamoro Autonomous Region In Muslim Mindanao (BARMM)': 'BARMM', 'MIMAROPA Region': 'Region IV-B', 'Negros Island Region (NIR)': 'NIR'}
LLM = globals().get('LLM', None)


def normalize_spaces(value):
    return re.sub(r'\s+', ' ', str(value or '')).strip()


def is_blank_like(value):
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    return normalize_spaces(value).lower() in NULL_LIKE_VALUES


def parse_coordinate(value, kind):
    if is_blank_like(value) or isinstance(value, bool):
        return None
    try:
        number = float(str(value).strip())
    except (TypeError, ValueError):
        return None
    if math.isnan(number) or math.isinf(number):
        return None
    if kind == LATITUDE_COLUMN and not -90 <= number <= 90:
        return None
    if kind == LONGITUDE_COLUMN and not -180 <= number <= 180:
        return None
    return number


def header_map(sheet):
    return {normalize_spaces(cell.value): cell.column for cell in sheet[1] if normalize_spaces(cell.value)}


def require_columns(sheet, names):
    headers = header_map(sheet)
    missing = [name for name in names if name not in headers]
    if missing:
        raise ValueError(f'Missing required exact column name(s): {missing}. Found: {list(headers)}')
    return headers


def last_header_column(sheet):
    return max([cell.column for cell in sheet[1] if normalize_spaces(cell.value)] or [0])


def append_fresh_columns(sheet, names):
    next_col = last_header_column(sheet) + 1
    output_cols = {}
    for name in names:
        sheet.cell(row=1, column=next_col).value = name
        output_cols[name] = next_col
        next_col += 1
    return output_cols


def load_file(path, sheet_name):
    if path.suffix.lower() == '.csv':
        workbook = Workbook()
        sheet = workbook.active
        sheet.title = path.stem[:31] or 'csv'
        with path.open('r', newline='', encoding='utf-8-sig') as handle:
            for row in csv.reader(handle):
                sheet.append(row)
        return workbook, sheet, 'csv'
    if path.suffix.lower() == '.xlsx':
        workbook = load_workbook(path)
        if sheet_name not in workbook.sheetnames:
            raise ValueError(f'Workbook does not contain sheet {sheet_name!r}. Available: {workbook.sheetnames}')
        return workbook, workbook[sheet_name], 'xlsx'
    raise ValueError('INPUT_PATH must end in .xlsx or .csv')


def save_file(workbook, sheet, kind, output_path):
    if kind == 'csv':
        with output_path.open('w', newline='', encoding='utf-8') as handle:
            writer = csv.writer(handle)
            for row in sheet.iter_rows(values_only=True):
                writer.writerow(['' if value is None else value for value in row])
    else:
        workbook.save(output_path)


def load_configured_file(input_path=None):
    path = Path(input_path or INPUT_PATH)
    if not str(path) or not path.exists():
        raise ValueError('Set INPUT_PATH to an existing .xlsx or .csv file path in the Variables cell.')
    return load_file(path, SHEET_NAME)


def compact_region_name(region_name):
    region_name = normalize_spaces(region_name)
    if region_name in SPECIAL_REGION_CODES:
        return SPECIAL_REGION_CODES[region_name]
    if region_name.startswith('Region '):
        return region_name.split(' (', 1)[0]
    raise ValueError(f'Unsupported PSGC region_name value: {region_name!r}')


def load_psgc_region_map(psgc_path=None):
    configured = str(psgc_path if psgc_path is not None else PSGC_PATH).strip()
    if not configured:
        raise ValueError('Set PSGC_PATH to philippines_psgc_barangays.csv in the Variables cell.')
    path = Path(configured)
    if not path.is_file():
        raise ValueError(f'PSGC_PATH does not exist or is not a file: {path}')
    if path.suffix.lower() != '.csv':
        raise ValueError('PSGC_PATH must point to the exported philippines_psgc_barangays.csv file.')
    province_regions = {}
    with path.open('r', newline='', encoding='utf-8-sig') as handle:
        reader = csv.DictReader(handle)
        required = {'province_name', 'region_name'}
        missing = sorted(required - set(reader.fieldnames or []))
        if missing:
            raise ValueError(f'PSGC CSV is missing required exact column name(s): {missing}. Found: {reader.fieldnames or []}')
        for row in reader:
            province = normalize_spaces(row.get('province_name'))
            region = normalize_spaces(row.get('region_name'))
            if province and region:
                province_regions.setdefault(province, set()).add(region)
    if not province_regions:
        raise ValueError('PSGC CSV did not contain any nonblank province_name and region_name pairs.')
    ambiguous = {province: sorted(regions) for province, regions in province_regions.items() if len(regions) != 1}
    if ambiguous:
        sample = list(ambiguous.items())[:5]
        raise ValueError(f'PSGC CSV maps a province to multiple regions: {sample}')
    mapping = {province: next(iter(regions)) for province, regions in province_regions.items()}
    for region in set(mapping.values()) | set(SPECIAL_PROVINCE_REGIONS.values()):
        compact_region_name(region)
    print(f'Loaded {len(mapping)} exact province-to-region mappings from: {path}')
    return mapping


def psgc_region_for_province(province, region_map):
    province = normalize_spaces(province)
    if not province:
        return None
    official_province = PROVINCE_ALIASES.get(province, province)
    region_name = SPECIAL_PROVINCE_REGIONS.get(official_province) or region_map.get(official_province)
    return compact_region_name(region_name) if region_name else None


def output_path_for_suffix(input_path, suffix):
    path = Path(input_path)
    if not str(path):
        raise ValueError('Set INPUT_PATH in the Variables cell.')
    if not suffix.startswith('_'):
        raise ValueError('Output suffix must start with an underscore, such as _coords, _parsed, or _cleaned.')
    return path.with_name(f'{path.stem}{suffix}{path.suffix}')


def save_configured_file(workbook, sheet, kind, output_suffix):
    path = output_path_for_suffix(INPUT_PATH, output_suffix)
    save_file(workbook, sheet, kind, path)
    return path


def geocode(address, api_key):
    response = requests.get('https://maps.googleapis.com/maps/api/geocode/json', params={'address': address, 'key': api_key}, timeout=30)
    response.raise_for_status()
    payload = response.json()
    if payload.get('status') != 'OK' or not payload.get('results'):
        return None
    location = payload['results'][0].get('geometry', {}).get('location', {})
    latitude = parse_coordinate(location.get('lat'), LATITUDE_COLUMN)
    longitude = parse_coordinate(location.get('lng'), LONGITUDE_COLUMN)
    return None if latitude is None or longitude is None else (latitude, longitude)


def add_missing_coordinates(workbook, sheet, kind, api_key=None):
    api_key = api_key or GOOGLE_GEOCODING_API_KEY
    if not api_key:
        raise ValueError('Set GOOGLE_GEOCODING_API_KEY in the Variables cell.')
    headers = require_columns(sheet, [LONGITUDE_COLUMN, LATITUDE_COLUMN, FULL_ADDRESS_COLUMN])
    longitude_col = headers[LONGITUDE_COLUMN]
    latitude_col = headers[LATITUDE_COLUMN]
    address_col = headers[FULL_ADDRESS_COLUMN]
    last_row = sheet.max_row if PROCESS_ROW_LIMIT is None else min(sheet.max_row, PROCESS_ROW_LIMIT + 1)
    summary = {'rows_checked': max(last_row - 1, 0), 'rows_needing_coordinates': 0, 'rows_geocoded': 0, 'skipped_blank_address_rows': [], 'no_result_rows': [], 'failed_rows': {}}

    for row_number in tqdm(range(2, last_row + 1), desc='Adding missing coordinates'):
        latitude_missing = parse_coordinate(sheet.cell(row=row_number, column=latitude_col).value, LATITUDE_COLUMN) is None
        longitude_missing = parse_coordinate(sheet.cell(row=row_number, column=longitude_col).value, LONGITUDE_COLUMN) is None
        if not (latitude_missing or longitude_missing):
            continue
        summary['rows_needing_coordinates'] += 1
        address = normalize_spaces(sheet.cell(row=row_number, column=address_col).value)
        if not address:
            summary['skipped_blank_address_rows'].append(row_number)
            continue
        try:
            result = geocode(address, api_key)
        except Exception as exc:
            summary['failed_rows'][row_number] = str(exc)
            continue
        if result is None:
            summary['no_result_rows'].append(row_number)
            continue
        latitude, longitude = result
        sheet.cell(row=row_number, column=latitude_col).value = latitude
        sheet.cell(row=row_number, column=longitude_col).value = longitude
        summary['rows_geocoded'] += 1
        time.sleep(0.05)
    summary['failed_row_count'] = len(summary.pop('failed_rows'))
    return summary


def extract_json_object_text(text):
    stripped = str(text or '').strip()
    if stripped.startswith('```'):
        stripped = re.sub(r'^```(?:json)?\s*', '', stripped, flags=re.I)
        stripped = re.sub(r'\s*```$', '', stripped)
    start, end = stripped.find('{'), stripped.rfind('}')
    if start < 0 or end < start:
        raise ValueError('No JSON object found in model response.')
    return stripped[start:end + 1]


def normalize_island_group(value):
    text = normalize_spaces(value).lower()
    if is_blank_like(text):
        return None
    if 'luzon' in text:
        return 'Luzon'
    if 'visayas' in text or 'visaya' in text:
        return 'Visayas'
    if 'mindanao' in text:
        return 'Mindanao'
    return None


def normalize_address_result(raw):
    if isinstance(raw, str):
        raw = json.loads(extract_json_object_text(raw))
    if not isinstance(raw, dict):
        raise ValueError('Model returned a non-object JSON value.')
    aliases = {'barangay': 'barangay', 'city_municipality': 'city_municipality', 'city/municipality': 'city_municipality', 'city': 'city_municipality', 'municipality': 'city_municipality', 'province': 'province', 'zip_code': 'zip_code', 'zip code': 'zip_code', 'zipcode': 'zip_code', 'postal_code': 'zip_code', 'country': 'country', 'island_group': 'island_group', 'island group': 'island_group', 'islandgroup': 'island_group'}
    normalized = {field: None for field in ADDRESS_FIELDS}
    for key, value in raw.items():
        canonical = aliases.get(normalize_spaces(key).lower())
        if canonical not in normalized:
            continue
        text = normalize_spaces(value)
        normalized[canonical] = normalize_island_group(text) if canonical == 'island_group' else (None if is_blank_like(text) else text)
    return normalized


def build_prompt(address):
    return f'''You parse Philippine addresses into administrative location fields.
Return only JSON with keys: barangay, city_municipality, province, zip_code, country, island_group.
Rules:
- Building names, company names, malls, roads, streets, drives, highways, subdivisions, landmarks, and branch names are not barangays, cities, or provinces.
- If barangay is not explicitly written as Brgy/Barangay or clearly known, use null.
- Never copy city_municipality into barangay unless the address explicitly marks that same value as Brgy/Barangay.
- ZIP code must be null unless a numeric postal code appears in the address.
- For Metro Manila cities, province is "Metro Manila".
- island_group must be exactly "Luzon", "Visayas", or "Mindanao"; use null if not confidently known.
- Country should be "Philippines" when the address is in the Philippines.
Example:
Address: Ropali Plaza, Escriva Drive, Pasig, Metro Manila
{{"barangay":null,"city_municipality":"Pasig","province":"Metro Manila","zip_code":null,"country":"Philippines","island_group":"Luzon"}}
Now parse this address:
{address}'''


def build_repair_prompt(address, previous_output):
    return f'''The previous response was not valid JSON. Return only one valid JSON object with exactly these keys: barangay, city_municipality, province, zip_code, country, island_group. Use null for missing values. island_group must be Luzon, Visayas, or Mindanao when known.
Address:
{address}
Invalid previous response:
{previous_output}'''


def model_generate(llm, prompt):
    messages = [{'role': 'system', 'content': 'You are a strict JSON extraction engine. Return only one compact JSON object. No markdown.'}, {'role': 'user', 'content': prompt}]
    output = llm.create_chat_completion(messages=messages, max_tokens=MAX_TOKENS, temperature=0)
    choice = output['choices'][0]
    message = choice.get('message') or {}
    return message.get('content') or choice.get('text') or ''


def parse_address(llm, address):
    first_output = model_generate(llm, build_prompt(address))
    try:
        return normalize_address_result(first_output)
    except Exception:
        return normalize_address_result(model_generate(llm, build_repair_prompt(address, first_output)))


def write_parsed_row(sheet, row_number, output_columns, parsed):
    for field, column_name in OUTPUT_FIELD_COLUMNS.items():
        sheet.cell(row=row_number, column=output_columns[column_name]).value = parsed.get(field)


def parse_full_address(workbook, sheet, kind, llm=None, psgc_region_map=None):
    llm = llm or globals().get('LLM')
    if llm is None:
        raise ValueError('LLM is not loaded. Run the Install and Check Llama cell first, then run Parse Full Address or Run Both.')
    psgc_region_map = psgc_region_map or load_psgc_region_map()
    headers = require_columns(sheet, [FULL_ADDRESS_COLUMN])
    address_col = headers[FULL_ADDRESS_COLUMN]
    output_columns = append_fresh_columns(sheet, PARSED_ADDRESS_COLUMNS)
    last_row = sheet.max_row if PROCESS_ROW_LIMIT is None else min(sheet.max_row, PROCESS_ROW_LIMIT + 1)
    summary = {'rows_checked': max(last_row - 1, 0), 'rows_parsed': 0, 'skipped_blank_address_rows': [], 'failed_rows': {}, 'region_rows_matched': 0, 'region_rows_unmatched': 0, 'unmatched_provinces': {}, 'appended_columns': PARSED_ADDRESS_COLUMNS}
    for row_number in tqdm(range(2, last_row + 1), desc='Parsing full address'):
        address = normalize_spaces(sheet.cell(row=row_number, column=address_col).value)
        if not address:
            summary['skipped_blank_address_rows'].append(row_number)
            parsed = {field: None for field in ADDRESS_FIELDS}
            parsed['region'] = None
            write_parsed_row(sheet, row_number, output_columns, parsed)
            unmatched = summary['unmatched_provinces'].setdefault('<blank>', {'count': 0, 'sample_rows': []})
            unmatched['count'] += 1
            if len(unmatched['sample_rows']) < 5:
                unmatched['sample_rows'].append(row_number)
            summary['region_rows_unmatched'] += 1
            continue
        try:
            parsed = parse_address(llm, address)
            summary['rows_parsed'] += 1
        except Exception as exc:
            summary['failed_rows'][row_number] = str(exc)
            parsed = {field: None for field in ADDRESS_FIELDS}
        parsed['region'] = psgc_region_for_province(parsed.get('province'), psgc_region_map)
        if parsed['region']:
            summary['region_rows_matched'] += 1
        else:
            province_key = normalize_spaces(parsed.get('province')) or '<blank>'
            unmatched = summary['unmatched_provinces'].setdefault(province_key, {'count': 0, 'sample_rows': []})
            unmatched['count'] += 1
            if len(unmatched['sample_rows']) < 5:
                unmatched['sample_rows'].append(row_number)
            summary['region_rows_unmatched'] += 1
        write_parsed_row(sheet, row_number, output_columns, parsed)
    summary['failed_row_count'] = len(summary.pop('failed_rows'))
    return summary

print('Shared helper functions loaded.')


## 1. Add Missing Longitude / Latitude

Uses `full_address` exactly as the Google Geocoding query and fills rows where `longitude` or `latitude` is blank, null-like, or invalid. Existing rows with both valid coordinates are not overwritten.


In [ ]:
workbook, sheet, kind = load_configured_file()
summary = add_missing_coordinates(workbook, sheet, kind)
saved_path = save_configured_file(workbook, sheet, kind, '_coords')
print('Saved:', saved_path)
print(json.dumps(summary, indent=2))


## 2. Parse Full Address

Uses `full_address`, the initialized `LLM`, and appends fresh parsed address columns. The LLM returns the province but not the region; `region2` is derived by exact province matching against the required PSGC CSV.


In [ ]:
psgc_region_map = load_psgc_region_map()

if globals().get('LLM') is None:
    from llama_cpp import Llama
    model_path = Path(GGUF_MODEL_PATH)
    if not GGUF_MODEL_PATH or not model_path.exists():
        raise ValueError('Set GGUF_MODEL_PATH to an existing .gguf model file path in the Variables cell.')
    print('Loading local model:', model_path)
    LLM = Llama(
        model_path=str(model_path),
        n_ctx=N_CTX,
        n_gpu_layers=N_GPU_LAYERS,
        flash_attn=FLASH_ATTN,
        verbose=VERBOSE_LLAMA,
    )

workbook, sheet, kind = load_configured_file()
summary = parse_full_address(workbook, sheet, kind, llm=LLM, psgc_region_map=psgc_region_map)
saved_path = save_configured_file(workbook, sheet, kind, '_parsed')
print('Saved:', saved_path)
print(json.dumps(summary, indent=2))


## 3. Run Both

Adds missing coordinates first, then parses full addresses, and saves one cleaned output file.


In [ ]:
psgc_region_map = load_psgc_region_map()
workbook, sheet, kind = load_configured_file()
coordinate_summary = add_missing_coordinates(workbook, sheet, kind)

if globals().get('LLM') is None:
    from llama_cpp import Llama
    model_path = Path(GGUF_MODEL_PATH)
    if not GGUF_MODEL_PATH or not model_path.exists():
        raise ValueError('Set GGUF_MODEL_PATH to an existing .gguf model file path in the Variables cell.')
    print('Loading local model:', model_path)
    LLM = Llama(
        model_path=str(model_path),
        n_ctx=N_CTX,
        n_gpu_layers=N_GPU_LAYERS,
        flash_attn=FLASH_ATTN,
        verbose=VERBOSE_LLAMA,
    )

parse_summary = parse_full_address(workbook, sheet, kind, llm=LLM, psgc_region_map=psgc_region_map)
saved_path = save_configured_file(workbook, sheet, kind, '_cleaned')
print('Saved:', saved_path)
print(json.dumps({'coordinates': coordinate_summary, 'full_address_parsing': parse_summary}, indent=2))
